### Init

#### Import

In [4]:
import os
import cv2
import dlib

import numpy as np
import pandas as pd
import tensorflow as tf

from PIL import Image
from tqdm import tqdm

from joblib import Parallel, delayed
from keras.applications.vgg16 import VGG16
from keras.preprocessing import image
from keras.applications.vgg16 import preprocess_input
from pytesseract import image_to_string
from tensorflow.keras.utils import load_img, img_to_array

#### Constants

In [40]:
# Specify cores to use
# -1 for all cores
N_JOBS = 2

### Utils

In [7]:
def open_image(path):
    img = None

    if os.path.exists(path):
        img = cv2.imread(path)
    else:
        print(f'Image not found: {path}')

    return img

In [ ]:
def process_image_kpis_parallel(image, funcs):
    # Apply each function to the image in parallel
    results = Parallel(n_jobs=N_JOBS)(delayed(func) for func in funcs)

    return results

### KPI functions

#### Colors

In [61]:
def avg_rgb_intensity(image):
    average_color_per_row = np.average(image, axis=0)
    average_color = np.average(average_color_per_row, axis=0)

    return [average_color[0], average_color[1], average_color[2]]

In [60]:
from sklearn.cluster import KMeans

def dominant_color(image):
    # Reshape the image to be a list of pixels
    pixels = image.reshape(-1, 3)

    # Fit a KMeans clustering model to the data
    kmeans = KMeans(n_clusters=5, n_init=10)
    kmeans.fit(pixels)

    # Count the number of pixels assigned to each cluster
    counts = np.bincount(kmeans.labels_)

    # The dominant color is the mean color of the most common cluster
    dominant = kmeans.cluster_centers_[counts.argmax()]

    colors = {
        "Red": [255, 0, 0],
        "Green": [0, 255, 0],
        "Blue": [0, 0, 255],
        "White": [255, 255, 255],
        "Black": [0, 0, 0]
    }

    return min(colors.keys(), key=lambda color: np.linalg.norm(dominant-colors[color]))

In [62]:
def num_distinct_colors(image):
    return len(set(tuple(v) for m2d in image for v in m2d))

In [63]:
def std_dev_color_intensities(image):
    # calculate standard deviation for each color channel
    std_dev_red = np.std(image[:,:,0])
    std_dev_green = np.std(image[:,:,1])
    std_dev_blue = np.std(image[:,:,2])

    return [std_dev_red, std_dev_green, std_dev_blue]

In [64]:
def color_intensity_peaks(image):
    # calculate histogram and find peak for each color channel
    hist_red = cv2.calcHist([image],[0],None,[256],[0,256])
    hist_green = cv2.calcHist([image],[1],None,[256],[0,256])
    hist_blue = cv2.calcHist([image],[2],None,[256],[0,256])

    return [np.argmax(hist_red), np.argmax(hist_green), np.argmax(hist_blue)]

In [65]:
def has_warm_colors(image):
    # define range for warm colors
    lower = np.array([0, 70, 50])
    upper = np.array([35, 255, 255])
    # convert image to HSV
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    # create a mask for warm colors
    mask = cv2.inRange(hsv, lower, upper)

    return int(np.any(mask))

In [66]:
def has_cool_colors(image):
    # define range for cool colors
    lower = np.array([60, 70, 50])
    upper = np.array([160, 255, 255])
    # convert image to HSV
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    # create a mask for cool colors
    mask = cv2.inRange(hsv, lower, upper)

    return int(np.any(mask))

#### Faces

In [14]:
def has_face_dlib(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    detector = dlib.get_frontal_face_detector()
    faces = detector(gray)

    return len(faces) > 0


In [15]:
def num_faces_dlib(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    detector = dlib.get_frontal_face_detector()
    faces = detector(gray)

    return len(faces)

In [16]:
def has_face_center(image):
    height, width, _ = image.shape
    center = image[height//4:3*height//4, width//4:3*width//4]
    faces = has_face_dlib(center)

    return faces

In [44]:
def faces_area_ratio(image):
    face_detector = dlib.get_frontal_face_detector()
    faces = face_detector(image, 1)
    total_faces_area = sum([rect.width() * rect.height() for rect in faces])
    total_image_area = image.shape[0] * image.shape[1]

    return total_faces_area / total_image_area

#### Text

In [17]:
def has_text_top_quarter(image):
    height, _, _ = image.shape
    top_quarter = image[:height//4, :]
    text = image_to_string(top_quarter)

    return 0 if text.strip() == "" else 1

In [18]:
def has_text_bottom_half(image):
    height, _, _ = image.shape
    bottom_half = image[height//2:, :]
    text = image_to_string(bottom_half)

    return 0 if text.strip() == "" else 1

In [19]:
def num_text_lines_bottom_quarter(image):
    height, _, _ = image.shape
    bottom_quarter = image[3*height//4:, :]
    text = image_to_string(bottom_quarter)
    lines = text.split("\n")

    return len(lines)

In [20]:
from pytesseract import pytesseract, Output

def largest_text_percentage(image):
    d = pytesseract.image_to_data(image, output_type=Output.DICT)
    n_boxes = len(d['level'])
    max_area = 0

    for i in range(n_boxes):
        (x, y, w, h) = (d['left'][i], d['top'][i], d['width'][i], d['height'][i])
        max_area = max(max_area, w * h)

    # return as percentage
    return (max_area * 100) / (image.shape[0] * image.shape[1])

In [21]:
def is_text_in_bottom_quarter(image):
    d = pytesseract.image_to_data(image, output_type=Output.DICT)
    n_boxes = len(d['level'])

    for i in range(n_boxes):
        (x, y, w, h) = (d['left'][i], d['top'][i], d['width'][i], d['height'][i])

        # If the center of the bounding box is in the bottom quarter
        if y + h/2 > image.shape[0] * 0.75:
            return 1

    return 0

In [22]:
def is_title_in_image(image, title):
    extracted_text = pytesseract.image_to_string(image)

    return int(title in extracted_text)

In [41]:
def largest_text_area_ratio(image):
    d = pytesseract.image_to_data(image, output_type=Output.DICT)
    n_boxes = len(d['level'])
    areas = [d['width'][i] * d['height'][i] for i in range(n_boxes)]
    largest_area = max(areas) if areas else 0
    total_area = image.shape[0] * image.shape[1]

    return largest_area / total_area

In [42]:
def has_central_text(image):
    central_region = image[int(image.shape[0] * 0.25):int(image.shape[0] * 0.75), int(image.shape[1] * 0.25):int(image.shape[1] * 0.75)]
    d = pytesseract.image_to_data(central_region, output_type=Output.DICT)

    return int(len(d['text']) > 0)

### Process kpi extraction

In [67]:
import warnings


warnings.filterwarnings("ignore")

In [68]:
img = open_image('5.jpg')

In [69]:
import json

funcs, keys = zip(*[
    # Color
    (dominant_color, 'dominant_color'),
    (has_warm_colors, 'has_warm_colors'),
    (has_cool_colors, 'has_cool_colors'),
    (avg_rgb_intensity, 'avg_rgb_intensity'),
    (num_distinct_colors, 'num_distinct_colors'),
    (std_dev_color_intensities, 'std_dev_color_intensities'),
    (color_intensity_peaks, 'color_intensity_peaks'),
    # # Facce
    # (has_face_dlib, 'has_face'),
    # (num_faces_dlib, 'num_faces'),
    # (has_face_center, 'has_face_center'),
    # (faces_area_ratio, 'faces_area_ratio'),
    # # Text
    # (has_central_text, 'has_central_text'),
    # (has_text_top_quarter, 'has_text_top_quarter'),
    # (has_text_bottom_half, 'has_text_bottom_half'),
    # (has_text_bottom_half, 'has_text_bottom_half'),
    # (largest_text_area_ratio, 'largest_text_area_ratio'),
    # (largest_text_percentage, 'largest_text_percentage'),
    # (is_text_in_bottom_quarter, 'is_text_in_bottom_quarter'),
    # (num_text_lines_bottom_quarter, 'num_text_lines_bottom_quarter'),
    # # Layout
    # (calculate_aspect_ratio, 'aspect_ratio'),
])

# Process image and get results
results = process_image_kpis_parallel(img, funcs)
[print(f'{type(x)} - {x}') for x in results]


# Combine keys and results into a dictionary
image_kpis = dict(zip(keys, results))


<class 'str'> - Black
<class 'int'> - 1
<class 'int'> - 1
<class 'list'> - [68.28104113924047, 88.08194936708857, 100.38561497890292]
<class 'int'> - 119688
<class 'list'> - [58.83137793381295, 69.55964234649977, 76.26760799966782]
<class 'list'> - [0, 1, 1]


In [75]:
def convert(o):
    if isinstance(o, np.int64):
        return int(o)
    raise TypeError

# Save the results to a JSON file
# with open('image_kpis.json', 'w') as file:
print(json.dumps(image_kpis, indent=4, default=convert))

{
    "dominant_color": "Black",
    "has_warm_colors": 1,
    "has_cool_colors": 1,
    "avg_rgb_intensity": [
        68.28104113924047,
        88.08194936708857,
        100.38561497890292
    ],
    "num_distinct_colors": 119688,
    "std_dev_color_intensities": [
        58.83137793381295,
        69.55964234649977,
        76.26760799966782
    ],
    "color_intensity_peaks": [
        0,
        1,
        1
    ]
}


---

### PAS MIS A JOUR
